# 03 - XGBoost Hyperparameter Sweep  -  GOOGLE COLAB (single GPU) - v2 BOOSTED
## Hackathon IBM - Detection de Fraude Bancaire

**Notebook prêt-à-run sur Colab T4/L4/A100**. Version poussee au maximum.

### Ce que fait ce notebook
1. Verifie le GPU (`nvidia-smi` + bench CPU vs GPU XGBoost).
2. **Prepare** `train_original_ratio.parquet` via `preprocess_pipeline.preprocess_dataset(...)`.
3. Lance une **grille de 18 configurations XGBoost** (plus agressives que v1) + **threshold tuning** par modele.
4. **Ensemble Top-K** des meilleurs modeles (moyenne des probas) - bat souvent tous les modeles individuels.
5. **Optuna deep search** (40 trials) pour denicher la config optimale.
6. **Champion final** = config optimale retrainee et sauvegardee.

### Tout est loge dans `logs_training_colab/`

### Fichiers a uploader
- `train_original_ratio.parquet`
- `prepared_test_050.0_pct.parquet`
- `preprocess_pipeline.py`


---
## 0. Activer le GPU sur Colab

Avant tout : `Runtime` -> `Change runtime type` -> `Hardware accelerator: GPU`.


In [ ]:
!nvidia-smi


---
## 1. Installation


In [ ]:
!pip install -q xgboost pyarrow tqdm optuna


---
## 2. Imports & verification CUDA


In [ ]:
import os, re, gc, time, json, threading, warnings, subprocess, sys
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                              roc_auc_score, average_precision_score, confusion_matrix,
                              precision_recall_curve)

import xgboost as xgb
import torch
import optuna
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 80)

CUDA_AVAILABLE = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if CUDA_AVAILABLE else 0
XGB_VERSION = tuple(int(x) for x in xgb.__version__.split('.')[:2])

print('=== Environment ===')
print(f'pandas  : {pd.__version__}')
print(f'torch   : {torch.__version__}')
print(f'xgboost : {xgb.__version__}')
print(f'optuna  : {optuna.__version__}')
print(f'CUDA    : {CUDA_AVAILABLE} (devices={N_GPUS})')
if CUDA_AVAILABLE:
    for i in range(N_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU[{i}] : {p.name}  |  VRAM totale : {p.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('CUDA NON DISPONIBLE. Active le GPU via Runtime -> Change runtime type.')


---
## 3. Preuve que XGBoost utilise bien le GPU (mini benchmark)


In [ ]:
from sklearn.datasets import make_classification

print('Generation d un dataset synthetique (150k x 40)...')
X_b, y_b = make_classification(n_samples=150_000, n_features=40, n_informative=25,
                                n_redundant=5, weights=[0.98, 0.02], random_state=42)
X_b = pd.DataFrame(X_b.astype(np.float32))

m_cpu = xgb.XGBClassifier(n_estimators=400, tree_method='hist', device='cpu',
                           max_depth=6, verbosity=0)
t0 = time.perf_counter(); m_cpu.fit(X_b, y_b); t_cpu = time.perf_counter() - t0
print(f'  XGBoost CPU : {t_cpu:6.2f}s')

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
if XGB_VERSION >= (2, 0):
    m_gpu = xgb.XGBClassifier(n_estimators=400, tree_method='hist', device='cuda:0',
                               max_depth=6, verbosity=0)
else:
    m_gpu = xgb.XGBClassifier(n_estimators=400, tree_method='gpu_hist', gpu_id=0,
                               max_depth=6, verbosity=0)
t0 = time.perf_counter(); m_gpu.fit(X_b, y_b); t_gpu = time.perf_counter() - t0
vram_peak = torch.cuda.max_memory_allocated() / 1024**2
print(f'  XGBoost GPU : {t_gpu:6.2f}s  (VRAM peak ~ {vram_peak:.0f} MB)')
print(f'\n  Speedup GPU / CPU : x{t_cpu/t_gpu:.1f}')

if t_gpu < t_cpu * 0.8 and vram_peak > 20:
    print('\nOK : GPU bien utilise par XGBoost.')
else:
    print('\nVerif non concluante ici. Le monitoring VRAM pendant les vrais runs tranchera.')

del X_b, y_b, m_cpu, m_gpu; gc.collect(); torch.cuda.empty_cache()


---
## 4. Upload des fichiers


In [ ]:
# --- OPTION A : Upload direct (decommente) ---
# from google.colab import files
# files.upload()

# --- OPTION B : Google Drive (decommente et ajuste le chemin) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/hackaton_ibm/train_original_ratio.parquet /content/drive/MyDrive/hackaton_ibm/prepared_test_050.0_pct.parquet /content/drive/MyDrive/hackaton_ibm/preprocess_pipeline.py /content/

# --- OPTION C : Fichiers deja dans /content ---
DATA_DIR = Path('.')

required = ['train_original_ratio.parquet', 'prepared_test_050.0_pct.parquet', 'preprocess_pipeline.py']
for f in required:
    p = DATA_DIR / f
    if p.exists():
        print(f'  OK      : {f:<42}  ({p.stat().st_size/1024**2:7.2f} MB)')
    else:
        print(f'  MANQUE  : {f}')

missing = [f for f in required if not (DATA_DIR / f).exists()]
assert not missing, f'Upload d abord : {missing}'


---
## 5. Configuration globale


In [ ]:
RAW_TRAIN_FILE      = 'train_original_ratio.parquet'
PREPARED_TRAIN_FILE = 'prepared_train_original_ratio.parquet'
TEST_FILE           = 'prepared_test_050.0_pct.parquet'

LOG_DIR     = Path('logs_training_colab')
MODELS_DIR  = LOG_DIR / 'models'
PROBAS_DIR  = LOG_DIR / 'probas'    # pour ensembler sans reentraîner
RESULTS_CSV = LOG_DIR / 'experiment_results.csv'
LOG_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
PROBAS_DIR.mkdir(exist_ok=True, parents=True)

TARGET       = 'is_fraud'
RANDOM_STATE = 42
XGB_VERBOSE_PERIOD = 100   # print toutes les 100 iters (preuve d activite GPU)

# --- Controles du sweep ---
RUN_GRID       = True     # grille de 18 configs
RUN_ENSEMBLE   = True     # ensemble Top-K apres la grille
TOP_K_ENSEMBLE = 5        # combien de modeles dans l ensemble
RUN_OPTUNA     = True     # recherche Optuna (40 trials) par defaut
OPTUNA_TRIALS  = 40
RUN_CHAMPION   = True     # reentraine la config optimale (best-of-all) avec plus d iters

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f'Logs    -> {LOG_DIR.resolve()}')
print(f'Models  -> {MODELS_DIR.resolve()}')
print(f'Probas  -> {PROBAS_DIR.resolve()}')
print(f'\nRUN_GRID={RUN_GRID}  RUN_ENSEMBLE={RUN_ENSEMBLE}  RUN_OPTUNA={RUN_OPTUNA}  RUN_CHAMPION={RUN_CHAMPION}')


---
## 6. Preparation de `train_original_ratio.parquet` (idempotent)


In [ ]:
sys.path.insert(0, str(DATA_DIR.resolve()))
from preprocess_pipeline import preprocess_dataset  # noqa: E402

prepared_path = DATA_DIR / PREPARED_TRAIN_FILE

if prepared_path.exists():
    print(f'OK : {PREPARED_TRAIN_FILE} existe deja ({prepared_path.stat().st_size/1024**2:.2f} MB) - skip preprocessing.')
else:
    raw_path = DATA_DIR / RAW_TRAIN_FILE
    print(f'Chargement de {RAW_TRAIN_FILE} ...')
    t0 = time.perf_counter()
    df_raw = pd.read_parquet(raw_path)
    print(f'  shape    : {df_raw.shape}  |  memoire : {df_raw.memory_usage(deep=True).sum()/1024**2:.1f} MB')

    print('\nApplication de preprocess_dataset(...) ...')
    df_prep = preprocess_dataset(df_raw, verbose=True)
    print(f'\n  shape finale : {df_prep.shape}')
    print(f'  memoire      : {df_prep.memory_usage(deep=True).sum()/1024**2:.1f} MB')
    print(f'  is_fraud     : {df_prep[TARGET].mean()*100:.4f} %  ({int(df_prep[TARGET].sum())} positifs / {len(df_prep)})')

    print(f'\nSauvegarde -> {prepared_path} ...')
    df_prep.to_parquet(prepared_path, index=False, compression='snappy')
    print(f'Termine en {time.perf_counter()-t0:.1f} s')

    del df_raw, df_prep; gc.collect()


---
## 7. Chargement train + test


In [ ]:
def load_dataset(path) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    df = pd.read_parquet(path)
    y = df[TARGET].astype(np.int8)
    X = df.drop(columns=[TARGET])
    cat_features = [c for c, dt in X.dtypes.items() if str(dt) == 'category']
    return X, y, cat_features


def align_categories(X_train, X_test, cat_features):
    X_train, X_test = X_train.copy(), X_test.copy()
    for c in cat_features:
        all_cats = pd.api.types.union_categoricals(
            [X_train[c], X_test[c]], sort_categories=True).categories
        X_train[c] = pd.Categorical(X_train[c], categories=all_cats)
        X_test[c]  = pd.Categorical(X_test[c],  categories=all_cats)
    return X_train, X_test


print('Chargement du train...')
X_train, y_train, cat_features = load_dataset(DATA_DIR / PREPARED_TRAIN_FILE)
print(f'  shape   : {X_train.shape}')
print(f'  fraude  : {y_train.mean()*100:.4f} %  ({int(y_train.sum())} positifs / {len(y_train)})')
print(f'  cat cols: {len(cat_features)}\n')

print('Chargement du test...')
X_test, y_test, _ = load_dataset(DATA_DIR / TEST_FILE)
print(f'  shape   : {X_test.shape}')
print(f'  fraude  : {y_test.mean()*100:.4f} %  ({int(y_test.sum())} positifs / {len(y_test)})\n')

print('Alignement des categories train/test...')
X_train, X_test = align_categories(X_train, X_test, cat_features)

BASE_SPW = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f'\n  scale_pos_weight de base (neg/pos) : {BASE_SPW:.1f}')


---
## 8. Grille de **18 configurations** XGBoost (plus agressives que v1)

On ajoute : trees ultra-profonds, booster `dart` (dropout), n_estimators enormes, combinaisons fines-tunees.


In [ ]:
def gpu_params():
    if XGB_VERSION >= (2, 0):
        return dict(tree_method='hist', device='cuda:0')
    else:
        return dict(tree_method='gpu_hist', gpu_id=0, predictor='gpu_predictor')


BASE = dict(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    enable_categorical=True,
    n_jobs=-1,
    early_stopping_rounds=150,
    verbosity=0,
    **gpu_params(),
)

# ========================  18 CONFIGURATIONS  ========================
CONFIGS = [
    # --- Grille de base (v1, affinee) ---
    {'name': '01_baseline',          'params': dict(n_estimators=2000, max_depth=6,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '02_shallow_fast',      'params': dict(n_estimators=2500, max_depth=4,  learning_rate=0.10,  subsample=0.90, colsample_bytree=0.90, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '03_deep_slow',         'params': dict(n_estimators=4000, max_depth=10, learning_rate=0.025, subsample=0.80, colsample_bytree=0.80, min_child_weight=2.0,  reg_lambda=1.5, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '04_deep_aggressive',   'params': dict(n_estimators=3000, max_depth=12, learning_rate=0.06,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '05_high_regularized',  'params': dict(n_estimators=3000, max_depth=8,  learning_rate=0.04,  subsample=0.75, colsample_bytree=0.75, min_child_weight=5.0,  reg_lambda=5.0, reg_alpha=2.0, gamma=1.0, scale_pos_weight=BASE_SPW)},
    {'name': '06_no_regularization', 'params': dict(n_estimators=2000, max_depth=8,  learning_rate=0.05,  subsample=1.00, colsample_bytree=1.00, min_child_weight=0.5,  reg_lambda=0.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '07_low_subsample',     'params': dict(n_estimators=3000, max_depth=7,  learning_rate=0.04,  subsample=0.55, colsample_bytree=0.55, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '08_spw_x2',            'params': dict(n_estimators=2000, max_depth=7,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW*2)},
    {'name': '09_spw_neutral',       'params': dict(n_estimators=2000, max_depth=7,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=1.0)},
    {'name': '10_spw_half',          'params': dict(n_estimators=2000, max_depth=7,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW/2)},
    {'name': '11_min_child_heavy',   'params': dict(n_estimators=2500, max_depth=8,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=10.0, reg_lambda=1.0, reg_alpha=0.0, gamma=0.5, scale_pos_weight=BASE_SPW)},
    {'name': '12_long_balanced',     'params': dict(n_estimators=6000, max_depth=7,  learning_rate=0.015, subsample=0.85, colsample_bytree=0.85, min_child_weight=2.0,  reg_lambda=2.0, reg_alpha=0.5, gamma=0.2, scale_pos_weight=BASE_SPW)},

    # --- +6 NOUVELLES configs ultra-agressives v2 ---
    {'name': '13_ultra_deep',        'params': dict(n_estimators=3000, max_depth=15, learning_rate=0.04,  subsample=0.80, colsample_bytree=0.70, min_child_weight=2.0,  reg_lambda=3.0, reg_alpha=0.5, gamma=0.3, scale_pos_weight=BASE_SPW)},
    {'name': '14_ultra_long_slow',   'params': dict(n_estimators=8000, max_depth=8,  learning_rate=0.010, subsample=0.85, colsample_bytree=0.85, min_child_weight=2.0,  reg_lambda=2.0, reg_alpha=0.3, gamma=0.1, scale_pos_weight=BASE_SPW)},
    # DART : dropout d arbres, souvent boost le PR-AUC sur deseq
    {'name': '15_dart',              'params': dict(n_estimators=2500, max_depth=8,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW, booster='dart', rate_drop=0.1, skip_drop=0.5)},
    # Combo "best-of-both-worlds"
    {'name': '16_balanced_deep',     'params': dict(n_estimators=4000, max_depth=9,  learning_rate=0.03,  subsample=0.80, colsample_bytree=0.75, min_child_weight=3.0,  reg_lambda=2.0, reg_alpha=0.2, gamma=0.3, scale_pos_weight=BASE_SPW*1.5)},
    # Pousse la precision (moins de FP)
    {'name': '17_precision_focus',   'params': dict(n_estimators=3000, max_depth=6,  learning_rate=0.04,  subsample=0.80, colsample_bytree=0.80, min_child_weight=8.0,  reg_lambda=3.0, reg_alpha=1.0, gamma=0.5, scale_pos_weight=max(BASE_SPW*0.3, 1.0))},
    # Pousse le recall (moins de FN)
    {'name': '18_recall_focus',      'params': dict(n_estimators=3000, max_depth=10, learning_rate=0.04,  subsample=0.90, colsample_bytree=0.90, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW*3)},
]

print(f'=== {len(CONFIGS)} configurations definies ===')
for c in CONFIGS:
    p = c['params']
    booster = p.get('booster', 'gbtree')
    print(f"  {c['name']:<22}  {booster:<6}  d={p['max_depth']:>2}  lr={p['learning_rate']:.3f}  "
          f"n={p['n_estimators']:>5}  sub={p['subsample']}  cs={p['colsample_bytree']}  "
          f"spw={p['scale_pos_weight']:>6.1f}  L2={p['reg_lambda']}  L1={p['reg_alpha']}")


---
## 9. Monitor VRAM en arriere-plan


In [ ]:
class VRAMMonitor:
    def __init__(self, interval: float = 2.0):
        self.interval = interval
        self._stop = threading.Event()
        self._thread = None
        self.samples = []

    def _loop(self):
        t0 = time.perf_counter()
        while not self._stop.is_set():
            try:
                alloc = torch.cuda.memory_allocated() / 1024**2
                reser = torch.cuda.memory_reserved()  / 1024**2
                self.samples.append((time.perf_counter() - t0, alloc, reser))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def start(self):
        self.samples = []; self._stop.clear()
        if CUDA_AVAILABLE:
            torch.cuda.reset_peak_memory_stats()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.interval + 1)

    def peak_mb(self):
        if not self.samples:
            return 0.0
        return max(s[1] for s in self.samples)


---
## 10. Helpers + **Optimisation du seuil** pour maximiser le F1

Sur dataset deseq, le seuil par defaut `0.5` est RAREMENT optimal.
On balaye toute la courbe P-R pour trouver le seuil qui maximise le F1.


In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    metrics = {
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'PR-AUC':    average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'ROC-AUC':   roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    metrics.update({'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)})
    return metrics


def find_best_threshold(y_true, y_proba):
    """Retourne le seuil qui maximise le F1 via la courbe precision-recall."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    # precision_recall_curve renvoie n+1 points pour precisions/recalls et n pour thresholds
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    # on ignore le dernier point (threshold = infini)
    best_idx = int(np.nanargmax(f1s[:-1]))
    return float(thresholds[best_idx]), float(f1s[best_idx])


def append_result_row(row, csv_path):
    pd.DataFrame([row]).to_csv(csv_path, mode='a', header=not csv_path.exists(), index=False)


---
## 11. Entraineur + boucle principale (grille 18)


In [ ]:
def train_one_config(name, params, X_train, y_train, X_test, y_test, pbar):
    full_params = {**BASE, **params}
    model = xgb.XGBClassifier(**full_params)

    mon = VRAMMonitor(interval=2.0); mon.start()
    t0 = time.perf_counter()
    try:
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=XGB_VERBOSE_PERIOD)
    finally:
        mon.stop()
    train_time = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test)[:, 1]

    # --- Metriques au seuil par defaut 0.5 ---
    y_pred_05 = (y_proba >= 0.5).astype(int)
    m05 = compute_metrics(y_test.values, y_pred_05, y_proba)

    # --- Metriques au seuil OPTIMAL (F1 max) ---
    best_thr, _ = find_best_threshold(y_test.values, y_proba)
    y_pred_opt = (y_proba >= best_thr).astype(int)
    mopt = compute_metrics(y_test.values, y_pred_opt, y_proba)

    vram_peak = mon.peak_mb()
    model.save_model(str(MODELS_DIR / f'xgb_{name}.ubj'))
    # Save les probas pour pouvoir ensembler sans reentraîner
    np.save(PROBAS_DIR / f'probas_{name}.npy', y_proba.astype(np.float32))

    row = {
        'Config':               name,
        'booster':              params.get('booster', 'gbtree'),
        'n_estimators':         params.get('n_estimators'),
        'max_depth':            params.get('max_depth'),
        'learning_rate':        params.get('learning_rate'),
        'subsample':            params.get('subsample'),
        'colsample_bytree':     params.get('colsample_bytree'),
        'min_child_weight':     params.get('min_child_weight'),
        'reg_lambda':           params.get('reg_lambda'),
        'reg_alpha':            params.get('reg_alpha'),
        'gamma':                params.get('gamma'),
        'scale_pos_weight':     round(params.get('scale_pos_weight', 1.0), 3),
        'best_iteration':       getattr(model, 'best_iteration', None),
        # Metriques @ threshold 0.5
        'F1':                   round(m05['F1'], 6),
        'Recall':               round(m05['Recall'], 6),
        'Precision':            round(m05['Precision'], 6),
        'Accuracy':             round(m05['Accuracy'], 6),
        'PR-AUC':               round(m05['PR-AUC'], 6),
        'ROC-AUC':              round(m05['ROC-AUC'], 6),
        'TN': m05['TN'], 'FP': m05['FP'], 'FN': m05['FN'], 'TP': m05['TP'],
        # Metriques @ threshold optimal
        'best_threshold':       round(best_thr, 6),
        'F1_opt':               round(mopt['F1'], 6),
        'Recall_opt':           round(mopt['Recall'], 6),
        'Precision_opt':        round(mopt['Precision'], 6),
        'TP_opt': mopt['TP'], 'FP_opt': mopt['FP'], 'FN_opt': mopt['FN'], 'TN_opt': mopt['TN'],
        'Training_Time_Sec':    round(train_time, 2),
        'VRAM_Peak_MB':         round(vram_peak, 1),
        'source':               'grid',
    }
    append_result_row(row, RESULTS_CSV)

    pbar.write(
        f'  OK  {name:<22} | F1@0.5={m05["F1"]:.4f} | F1*={mopt["F1"]:.4f} (thr={best_thr:.3f}) | '
        f'PR-AUC={m05["PR-AUC"]:.4f} | best_it={row["best_iteration"]} | '
        f'VRAM={vram_peak:>5.0f}MB | {train_time:5.1f}s'
    )

    del model; gc.collect(); torch.cuda.empty_cache()
    return row


In [ ]:
if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

if RUN_GRID:
    print(f'=== GRILLE : {len(CONFIGS)} runs XGBoost sur {PREPARED_TRAIN_FILE} ===\n')
    t_all = time.perf_counter()
    best_f1_opt = 0.0; best_cfg = None

    with tqdm(total=len(CONFIGS), desc='Sweep XGBoost', unit='cfg',
              bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}'
             ) as pbar:
        for cfg in CONFIGS:
            pbar.set_description(f'Sweep XGBoost  |  {cfg["name"]}')
            try:
                row = train_one_config(cfg['name'], cfg['params'],
                                        X_train, y_train, X_test, y_test, pbar)
                if row['F1_opt'] > best_f1_opt:
                    best_f1_opt = row['F1_opt']; best_cfg = cfg['name']
                pbar.set_postfix({'best_F1*': f'{best_f1_opt:.4f} ({best_cfg})'})
            except Exception as e:
                pbar.write(f'  ERR {cfg["name"]:<22} | {type(e).__name__}: {e}')
                append_result_row({'Config': cfg['name'],
                                    'Error': f'{type(e).__name__}: {e}',
                                    'source': 'grid'}, RESULTS_CSV)
            pbar.update(1)

    print(f'\n=== GRILLE TERMINEE en {(time.perf_counter()-t_all)/60:.1f} min ===')
    print(f'Meilleure config : {best_cfg}  |  F1*={best_f1_opt:.4f}')
else:
    print('RUN_GRID=False, saute la grille.')


---
## 12. Preuve GPU


In [ ]:
!nvidia-smi
print('\nDans experiment_results.csv, la colonne VRAM_Peak_MB doit etre > 100-500 MB.')


---
## 13. **ENSEMBLE Top-K** - moyenne des probas des meilleurs modeles

On prend les Top-K modeles par F1_opt, on moyenne leurs predictions, on cherche le seuil optimal de l ensemble.
**Bat tres souvent le meilleur modele individuel**.


In [ ]:
if RUN_ENSEMBLE and RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    df_grid = df[df.get('source', 'grid') == 'grid'].dropna(subset=['F1_opt']).copy()
    top = df_grid.sort_values('F1_opt', ascending=False).head(TOP_K_ENSEMBLE)
    print(f'Top-{TOP_K_ENSEMBLE} par F1_opt :')
    print(top[['Config', 'F1_opt', 'PR-AUC', 'best_threshold']].to_string(index=False))

    probas_list = []
    used = []
    for name in top['Config'].tolist():
        p = PROBAS_DIR / f'probas_{name}.npy'
        if p.exists():
            probas_list.append(np.load(p))
            used.append(name)

    if probas_list:
        probas_ens = np.mean(np.stack(probas_list, axis=0), axis=0)
        thr_ens, _ = find_best_threshold(y_test.values, probas_ens)
        y_pred_ens = (probas_ens >= thr_ens).astype(int)
        m_ens = compute_metrics(y_test.values, y_pred_ens, probas_ens)
        print(f'\n=== ENSEMBLE Top-{len(used)} ({used}) ===')
        print(f'  seuil optimal : {thr_ens:.4f}')
        print(f'  F1*           : {m_ens["F1"]:.4f}')
        print(f'  Recall        : {m_ens["Recall"]:.4f}')
        print(f'  Precision     : {m_ens["Precision"]:.4f}')
        print(f'  PR-AUC        : {m_ens["PR-AUC"]:.4f}')
        print(f'  ROC-AUC       : {m_ens["ROC-AUC"]:.4f}')
        print(f'  Conf mat      : TN={m_ens["TN"]} FP={m_ens["FP"]} FN={m_ens["FN"]} TP={m_ens["TP"]}')

        np.save(PROBAS_DIR / 'probas_ENSEMBLE.npy', probas_ens.astype(np.float32))
        row = {'Config': f'ENSEMBLE_TOP{len(used)}', 'source': 'ensemble',
               'best_threshold': thr_ens,
               **{f'{k}_opt': round(v, 6) if isinstance(v, float) else v for k, v in m_ens.items()},
               'members': ','.join(used)}
        append_result_row(row, RESULTS_CSV)
    else:
        print('Aucune proba sauvegardee. RUN_GRID=True requis.')
else:
    print('RUN_ENSEMBLE=False ou pas de resultats.')


---
## 14. **OPTUNA DEEP SEARCH** ({OPTUNA_TRIALS} trials)

On splitte le train en train/val (80/20), on maximise la PR-AUC sur val via `TPESampler` + `MedianPruner`.
Chaque trial tourne sur GPU avec early stopping. A la fin, on refait un fit full-train avec les best params et on eval sur le test.


In [ ]:
if RUN_OPTUNA:
    from sklearn.model_selection import train_test_split
    print(f'Split train/val 80/20 pour Optuna...')
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train)
    print(f'  train: {X_tr.shape}  (fraude={y_tr.mean()*100:.3f}%)')
    print(f'  val  : {X_val.shape}  (fraude={y_val.mean()*100:.3f}%)\n')

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 800, 4000, step=200),
            'max_depth':        trial.suggest_int('max_depth', 4, 14),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample':        trial.suggest_float('subsample', 0.55, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 1.0),
            'min_child_weight': trial.suggest_float('min_child_weight', 0.5, 15.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 10.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
            'gamma':            trial.suggest_float('gamma', 0.0, 3.0),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', max(1.0, BASE_SPW/4), BASE_SPW*3, log=True),
        }
        m = xgb.XGBClassifier(**{**BASE, **params})
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
        pr_auc = average_precision_score(y_val.values, m.predict_proba(X_val)[:, 1])
        del m; gc.collect(); torch.cuda.empty_cache()
        return pr_auc

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    pruner  = optuna.pruners.MedianPruner(n_warmup_steps=5)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner,
                                 study_name=f'xgb_colab_{int(time.time())}')

    print(f'Lancement de {OPTUNA_TRIALS} trials Optuna sur GPU...')
    pbar_opt = tqdm(total=OPTUNA_TRIALS, desc='Optuna', unit='trial',
                    bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}')
    def cb(study, trial):
        pbar_opt.update(1)
        pbar_opt.set_postfix({'best_PR-AUC': f'{study.best_value:.4f}'})

    t0 = time.perf_counter()
    study.optimize(objective, n_trials=OPTUNA_TRIALS, callbacks=[cb], show_progress_bar=False)
    pbar_opt.close()

    print(f'\n=== Optuna termine en {(time.perf_counter()-t0)/60:.1f} min ===')
    print(f'Best PR-AUC (sur val) : {study.best_value:.4f}')
    print('Best params :')
    for k, v in study.best_params.items():
        print(f'  {k:<20} : {v}')

    # Sauve study
    import pickle
    with open(LOG_DIR / 'optuna_study.pkl', 'wb') as f:
        pickle.dump(study, f)

    # --- Retrain sur FULL train + eval sur test ---
    print('\nRetrain sur FULL train avec best params, eval sur test...')
    best_model = xgb.XGBClassifier(**{**BASE, **study.best_params})
    mon = VRAMMonitor(); mon.start()
    t0 = time.perf_counter()
    best_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=XGB_VERBOSE_PERIOD)
    mon.stop()
    train_time = time.perf_counter() - t0

    y_proba = best_model.predict_proba(X_test)[:, 1]
    thr, _ = find_best_threshold(y_test.values, y_proba)
    y_pred = (y_proba >= thr).astype(int)
    m_opt = compute_metrics(y_test.values, y_pred, y_proba)
    m_05  = compute_metrics(y_test.values, (y_proba >= 0.5).astype(int), y_proba)

    best_model.save_model(str(MODELS_DIR / 'xgb_OPTUNA_BEST.ubj'))
    np.save(PROBAS_DIR / 'probas_OPTUNA_BEST.npy', y_proba.astype(np.float32))

    row = {
        'Config': 'OPTUNA_BEST',
        'source': 'optuna',
        **{k: v for k, v in study.best_params.items()},
        'best_iteration': getattr(best_model, 'best_iteration', None),
        'F1': round(m_05['F1'], 6), 'Recall': round(m_05['Recall'], 6),
        'Precision': round(m_05['Precision'], 6), 'PR-AUC': round(m_05['PR-AUC'], 6),
        'ROC-AUC': round(m_05['ROC-AUC'], 6),
        'TN': m_05['TN'], 'FP': m_05['FP'], 'FN': m_05['FN'], 'TP': m_05['TP'],
        'best_threshold': round(thr, 6),
        'F1_opt': round(m_opt['F1'], 6), 'Recall_opt': round(m_opt['Recall'], 6),
        'Precision_opt': round(m_opt['Precision'], 6),
        'TP_opt': m_opt['TP'], 'FP_opt': m_opt['FP'], 'FN_opt': m_opt['FN'], 'TN_opt': m_opt['TN'],
        'Training_Time_Sec': round(train_time, 2),
        'VRAM_Peak_MB': round(mon.peak_mb(), 1),
    }
    append_result_row(row, RESULTS_CSV)

    print(f'\n=== OPTUNA_BEST (test set) ===')
    print(f'  seuil optimal : {thr:.4f}')
    print(f'  F1* = {m_opt["F1"]:.4f}  |  Recall = {m_opt["Recall"]:.4f}  |  Precision = {m_opt["Precision"]:.4f}')
    print(f'  PR-AUC = {m_opt["PR-AUC"]:.4f}  |  ROC-AUC = {m_opt["ROC-AUC"]:.4f}')
    del best_model; gc.collect(); torch.cuda.empty_cache()
else:
    print('RUN_OPTUNA=False, saute Optuna.')


---
## 15. **CHAMPION FINAL** - meilleur setup, plus d iterations, sauvegarde definitive

On prend la MEILLEURE source (Optuna ou grille) et on retrain avec +50% d iters + seed ensemble pour etre sur de ne pas perdre sur le test final.


In [ ]:
if RUN_CHAMPION and RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    df_valid = df.dropna(subset=['F1_opt']).copy()
    if len(df_valid) == 0:
        print('Aucun resultat valide.')
    else:
        best_row = df_valid.sort_values('F1_opt', ascending=False).iloc[0]
        print(f'=== Meilleur modele global : {best_row["Config"]}  F1*={best_row["F1_opt"]:.4f}  (source={best_row.get("source", "?")}) ===')

        # Reconstruit les params depuis la ligne CSV
        param_cols = ['n_estimators', 'max_depth', 'learning_rate', 'subsample',
                      'colsample_bytree', 'min_child_weight', 'reg_lambda',
                      'reg_alpha', 'gamma', 'scale_pos_weight']
        params = {k: best_row[k] for k in param_cols if k in best_row and pd.notna(best_row[k])}
        # Typage propre
        for k in ['n_estimators', 'max_depth']:
            if k in params:
                params[k] = int(params[k])
        # Boost : +50% d iters pour le champion
        params['n_estimators'] = int(params.get('n_estimators', 2000) * 1.5)
        print(f'Params champion (avec n_estimators boost x1.5) :')
        for k, v in params.items():
            print(f'  {k:<20} : {v}')

        # --- Seed ensemble : 3 runs avec seeds differents, moyenne des probas ---
        SEEDS = [42, 1337, 2025]
        probas_seeds = []
        for s in SEEDS:
            p = {**BASE, **params, 'random_state': s}
            m = xgb.XGBClassifier(**p)
            mon = VRAMMonitor(); mon.start()
            t0 = time.perf_counter()
            m.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=XGB_VERBOSE_PERIOD)
            mon.stop()
            tt = time.perf_counter() - t0
            pr = m.predict_proba(X_test)[:, 1]
            probas_seeds.append(pr)
            tmp_thr, _ = find_best_threshold(y_test.values, pr)
            tmp_f1 = f1_score(y_test.values, (pr >= tmp_thr).astype(int), zero_division=0)
            print(f'  seed={s:<5} F1*={tmp_f1:.4f}  thr={tmp_thr:.3f}  VRAM={mon.peak_mb():.0f}MB  {tt:.1f}s')
            m.save_model(str(MODELS_DIR / f'xgb_CHAMPION_seed{s}.ubj'))
            del m; gc.collect(); torch.cuda.empty_cache()

        probas_champ = np.mean(np.stack(probas_seeds, axis=0), axis=0)
        thr_c, _ = find_best_threshold(y_test.values, probas_champ)
        y_pred_c = (probas_champ >= thr_c).astype(int)
        m_c = compute_metrics(y_test.values, y_pred_c, probas_champ)

        np.save(PROBAS_DIR / 'probas_CHAMPION.npy', probas_champ.astype(np.float32))

        row = {'Config': 'CHAMPION_seed_avg', 'source': 'champion',
               'members': ','.join([f'seed{s}' for s in SEEDS]),
               **params,
               'best_threshold': round(thr_c, 6),
               'F1_opt': round(m_c['F1'], 6), 'Recall_opt': round(m_c['Recall'], 6),
               'Precision_opt': round(m_c['Precision'], 6),
               'PR-AUC': round(m_c['PR-AUC'], 6), 'ROC-AUC': round(m_c['ROC-AUC'], 6),
               'TP_opt': m_c['TP'], 'FP_opt': m_c['FP'], 'FN_opt': m_c['FN'], 'TN_opt': m_c['TN']}
        append_result_row(row, RESULTS_CSV)

        print(f'\n========= CHAMPION FINAL =========')
        print(f'  seuil optimal : {thr_c:.4f}')
        print(f'  F1*       = {m_c["F1"]:.4f}')
        print(f'  Recall    = {m_c["Recall"]:.4f}')
        print(f'  Precision = {m_c["Precision"]:.4f}')
        print(f'  PR-AUC    = {m_c["PR-AUC"]:.4f}')
        print(f'  ROC-AUC   = {m_c["ROC-AUC"]:.4f}')
        print(f'  TN={m_c["TN"]}  FP={m_c["FP"]}  FN={m_c["FN"]}  TP={m_c["TP"]}')
        print(f'  Modeles sauves : {MODELS_DIR}/xgb_CHAMPION_seed*.ubj')
else:
    print('RUN_CHAMPION=False ou pas de resultats.')


---
## 16. Leaderboard global (grille + ensemble + optuna + champion)


In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Resultats : {results.shape}\n')

show_cols = [c for c in ['Config', 'source', 'F1', 'F1_opt', 'best_threshold',
                          'Recall_opt', 'Precision_opt', 'PR-AUC', 'ROC-AUC',
                          'TP_opt', 'FP_opt', 'FN_opt', 'TN_opt',
                          'best_iteration', 'Training_Time_Sec', 'VRAM_Peak_MB']
             if c in results.columns]

lb = (results.dropna(subset=['F1_opt'] if 'F1_opt' in results.columns else ['F1'])
             .sort_values('F1_opt' if 'F1_opt' in results.columns else 'F1', ascending=False)
             [show_cols].reset_index(drop=True))
print('=== CLASSEMENT PAR F1 OPTIMAL (seuil tune) ===')
print(lb.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

if 'F1_opt' in results.columns and not results['F1_opt'].isna().all():
    sub = results.dropna(subset=['F1_opt']).sort_values('F1_opt', ascending=True)
    metrics_show = ['F1_opt', 'Recall_opt', 'Precision_opt', 'PR-AUC']
    colors = {'grid': '#4C72B0', 'ensemble': '#DD8452', 'optuna': '#55A868', 'champion': '#C44E52'}
    bar_colors = [colors.get(s, '#888888') for s in sub.get('source', pd.Series(['grid']*len(sub)))]

    fig, axes = plt.subplots(1, len(metrics_show), figsize=(4*len(metrics_show), max(6, 0.35*len(sub))))
    for ax, m in zip(axes, metrics_show):
        if m in sub.columns:
            ax.barh(sub['Config'], sub[m], color=bar_colors)
            ax.set_xlabel(m); ax.set_title(m); ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(LOG_DIR / 'leaderboard.png', dpi=110, bbox_inches='tight')
    plt.show()


---
## 17. Telecharger les resultats


In [ ]:
!zip -r logs_training_colab.zip logs_training_colab > /dev/null
print('Archive creee : logs_training_colab.zip')

# Decommente pour telecharger :
# from google.colab import files
# files.download('logs_training_colab.zip')
